In [3]:
import torch
from model import *
from torchinfo import summary
from PIL import Image
import torchvision.transforms as T
import numpy as np
from people_dataset import *
from loss_function import *
import torch.nn as nn
import argparse
from torch.utils.data import DataLoader,Dataset
import gc
import time
import matplotlib.pyplot as plt
import torch.nn.functional as F
import sys
import json
from model import * 
from person_defom_stereo import denormalize,InputPadder
sys.path.append('core')
from core.defom_stereo import DEFOMStereo
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
def load_last_checkpoint_weights(model,path,device):
    if os.path.exists(path):
        state_dict=torch.load('models/right_generator/stereo_generator_final.pth',map_location=device,weights_only=True )
        model.load_state_dict(state_dict)
        print("模型成功加载了上一个检查点")
    else:
        print("模型重新开始训练")

In [5]:
class Parameters_DEFOMStereo():
    def __init__(self):
        self.parser = argparse.ArgumentParser()
        self.parser.add_argument('--mixed_precision', action='store_true', help='use mixed precision')
        self.parser.add_argument('--valid_iters', type=int, default=32, help='number of flow-field updates during forward pass')
        self.parser.add_argument('--scale_iters', type=int, default=8, help="number of scaling updates to the disparity field in each forward pass.")
        # DefomStereo Architecture choices
        self.parser.add_argument('--dinov2_encoder', type=str, default='vits', choices=['vits', 'vitb', 'vitl', 'vitg'])
        self.parser.add_argument('--idepth_scale', type=float, default=0.5, help="the scale of inverse depth to initialize disparity")
        self.parser.add_argument('--hidden_dims', nargs='+', type=int, default=[128]*3, help="hidden state and context dimensions")
        self.parser.add_argument('--corr_implementation', choices=["reg", "alt", "reg_cuda", "alt_cuda"], default="reg", help="correlation volume implementation")
        self.parser.add_argument('--shared_backbone', action='store_true', help="use a single backbone for the context and feature encoders")
        self.parser.add_argument('--corr_levels', type=int, default=2, help="number of levels in the correlation pyramid")
        self.parser.add_argument('--corr_radius', type=int, default=4, help="width of the correlation pyramid")
        self.parser.add_argument('--scale_list', type=float, nargs='+', default=[0.125, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
                            help='the list of scaling factors of disparity')
        self.parser.add_argument('--scale_corr_radius', type=int, default=2,
                            help="width of the correlation pyramid for scaled disparity")
        self.parser.add_argument('--n_downsample', type=int, default=2, choices=[2, 3], help="resolution of the disparity field (1/2^K)")
        self.parser.add_argument('--context_norm', type=str, default="batch", choices=['group', 'batch', 'instance', 'none'], help="normalization of context encoder")
        self.parser.add_argument('--n_gru_layers', type=int, default=3, help="number of hidden GRU levels")
    def parse(self):
        self.args,_=self.parser.parse_known_args()
        return self.args

In [6]:
DEFOMStereo_parser=Parameters_DEFOMStereo()
args=DEFOMStereo_parser.parse()

In [7]:
model_predict_right=GeometryAwareStereoGenerator()
model_predict_disp=DEFOMStereo(args)
model_predict_disp.load_state_dict(torch.load('models/defom_stereo/defomstereo_vits_sceneflow.pth',map_location=device,weights_only=True ))

<All keys matched successfully>

In [8]:
dataset=PeopleDataSet(have_disp=True)
dataloader=DataLoader(dataset,4,shuffle=True,num_workers=4,drop_last=False)
criterion=StereoLoss().to(device)
optimizer = torch.optim.Adam(model_predict_right.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.6)

In [ ]:
for lefts,rights,disps in dataloader:
    rights_pre=model_predict_right(lefts)
    padder = InputPadder(lefts.shape, divis_by=32)
    lefts, rights = padder.pad(lefts, rights)
    lefts=denormalize(lefts)
    rights=denormalize(rights,mean=[0],std=[1])
    disps_pre=model_predict_disp(lefts,rights,test_mode=False)
    print(disps_pre.shape)
    #loss=criterion(right_pre,rights,disps)
    
    break